The Decoder module generates local high-resolution forecast maps (1 km resolution) from the
coarse predictions. Even if the ConvLSTM/Processor operates on a lower-resolution grid (for
computational efficiency or due to patch embedding), the Decoder will upsample and refine the output.

U-Net architecture, a well-known convolutional network with an encoder-decoder structure and skip connections that preserve fine details . The U-Net takes as input the coarse forecast (e.g. a 2D field at, say, 10 km resolution) and outputs a finer 1 km grid.

-----------------------------------------------------------------------------------------------------------------------------
this acts as a super
resolution or downscaling model, adding local detail (potentially informed by high-res static data like
 topography or coastline, if we include those as additional inputs).

Our U-Net Decoder operates per forecast time step (it processes one frame at a time, independently,
 since spatial super-resolution can be learned time-independently). We design the U-Net with a
 contracting path that reduces the spatial dimension and an expanding path that increases it back, with
 skip connections from contracting to expanding path to preserve high-frequency information.

In [ ]:
class UNetDecoder(nn.Module):
def __init__(self, in_channels, out_channels, base_channels=64):
    super().__init__()
    
    # Contracting path
    self.enc1 = nn.Sequential(
    nn.Conv2d(in_channels, base_channels, 3, padding=1), nn.ReLU(),
    nn.Conv2d(base_channels, base_channels, 3, padding=1), nn.ReLU())
    self.pool1 = nn.MaxPool2d(2)
    self.enc2 = nn.Sequential(
    nn.Conv2d(base_channels, base_channels*2, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(base_channels*2, base_channels*2, 3, padding=1),
    nn.ReLU())
    self.pool2 = nn.MaxPool2d(2)
    self.enc3 = nn.Sequential(
    nn.Conv2d(base_channels*2, base_channels*4, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(base_channels*4, base_channels*4, 3, padding=1),
    nn.ReLU())
    
    # Expanding path
    self.up2 = nn.ConvTranspose2d(base_channels*4, base_channels*2, kernel_size=2, stride=2)
    self.dec2 = nn.Sequential(
    nn.Conv2d(base_channels*4, base_channels*2, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(base_channels*2, base_channels*2, 3, padding=1),
    nn.ReLU())
    self.up1 = nn.ConvTranspose2d(base_channels*2, base_channels, kernel_size=2, stride=2)
    self.dec1 = nn.Sequential(nn.Conv2d(base_channels*2, base_channels, 3, padding=1),
    nn.ReLU(),
    nn.Conv2d(base_channels, base_channels, 3, padding=1), nn.ReLU())
    self.final = nn.Conv2d(base_channels, out_channels, kernel_size=1)
 def forward(self, x):
    
    # x: (B, in_channels, H_coarse, W_coarse)
    e1 = self.enc1(x) # (B, 64, H, W)
    p1 = self.pool1(e1) # (B, 64, H/2, W/2)
    e2 = self.enc2(p1) # (B, 128, H/2, W/2)
    p2 = self.pool2(e2) # (B, 128, H/4, W/4)
    e3 = self.enc3(p2) # (B, 256, H/4, W/4)
    
    # Decoder
    u2 = self.up2(e3) # (B, 128, H/2, W/2)
    u2 = torch.cat([u2, e2], dim=1) # skip connection concatenation
    d2 = self.dec2(u2) # (B, 128, H/2, W/2)
    u1 = self.up1(d2) # (B, 64, H, W)
    u1 = torch.cat([u1, e1], dim=1) # concat skip from e1
    d1 = self.dec1(u1) # (B, 64, H, W)
    out = self.final(d1)
    return out
# (B, out_channels, H, W)

 We integrate Encoder, Processor, and Decoder into one end-to-end model, which we’ll call
 WeatherForecastNet . During training, the dataflow is:
 1. 
2. 
3. 
4. 
Input: A sequence of past observations (or just the current state) on the common grid. For
 example, we might use the last 3 hourly grids as input to provide the model some notion of
 recent motion, although the problem statement focuses on assimilating current data.
 Encoder: Produces encoded features (global and/or spatial) from the input.
 Processor: Generates a sequence of coarse future predictions (up to 48 frames for 48 hours).
 Decoder: Upscales each coarse frame to high resolution.
 We train the model in a supervised manner using historical data. We need training pairs of (input data,
 ground-truth future data). Ground truth could come from reanalysis (like ERA5) or from the actual
 observations at +hours (for example, satellite images at future times, buoy readings at future times,
 etc., interpolated to the grid).

 Training Loop: We utilize PyTorch’s training paradigm with an optimizer like Adam:

In [ ]:
import torch.optim as optim
model = WeatherForecastNet() # which internally contains Encoder, ConvLSTM, Decoder
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader: # assume train_loader yields (input_tensor, target_tensor)
        input_tensor, target_tensor = batch
        # target_tensor shape: (B, T=48, out_ch, H, W)
        pred = model(input_tensor)
        # pred shape matches target
        loss = loss_fn(pred, target_tensor)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Validation loop (if any) would go here

Model Export (ONNX/TorchScript) for Deployment
 After training, we want to deploy the model in a production environment (Rust). We will export the
 PyTorch model to a format suitable for fast inference. Two common choices are ONNX (Open Neural
 Network Exchange format) and TorchScript.

 

_Model Export (ONNX/TorchScript) for Deployment_

 After training, we want to deploy the model in a production environment (Rust). We will export the
 PyTorch model to a format suitable for fast inference. Two common choices are ONNX (Open Neural
 Network Exchange format) and TorchScript.

ONNX Export: We can export the model to 
model.onnx using 
torch.onnx.export . This
 requires a sample input and converts the static graph to ONNX. We should specify 
opset_version (e.g. 13) and ensure all components are compatible (PyTorch’s modules like
 Transformer, Conv, etc. have ONNX support). ONNX is advantageous because we can use the
 highly optimized ONNX Runtime in Rust for inference

In [ ]:
model.eval()
dummy_input = torch.randn(1, 3, 6, NY, NX) # e.g. batch=1, time=3, 6 channels, grid size
 torch.onnx.export(model, dummy_input, "forecast_net.onnx", opset_version=13, input_names=['input'], output_names=['output'], dynamic_axes={'input': {0: 'batch', 1: 'time'}, 'output': {0: 'batch', 1: 'time'}}

_Rust Implementation – Data Pipeline and Optimized Inference_
In production, we implement the backend in Rust for performance and reliability. There are two main
components in Rust:
- 1. Data Fetch & Preprocessing Module (Rust),  Below is a conceptual snippet of Rust code for data fetching and grid assembly: 

In [ ]:
 use reqwest;
 use image;
 use ndarray::{Array3, Array2};
 use serde::Deserialize;
 #[derive(Deserialize)]
 struct BuoyObs { latitude: f64, longitude: f64, value: f32 }
 fn fetch_and_preprocess()-> Array3<f32> {
    // Grid parameters
    let lat_min = 30.0;
    let lon_min =-5.0;
    let ny = ((45.0- lat_min) / 0.01) as usize;
    let nx = ((15.0- lon_min) / 0.01) as usize;
    let mut grid = Array3::<f32>::zeros((6, ny, nx)); // 6 channels as example
 // 1. Fetch satellite image
 let sat_img_bytes = reqwest::blocking::get("https://example.com/noaa_apt_latest.jpg").expect("Failed to fetch image").bytes().expect("No bytes");
 let sat_img = image::load_from_memory(&sat_img_bytes).expect("Invalid image").to_luma8();
 // Resize image to grid dimensions using image crate
 let sat_img_resized = image::imageops::resize(&sat_img, nx as u32, ny as
 u32, image::imageops::Nearest);
 // Place into grid channel 0
 for (y, x, pixel) in sat_img_resized.enumerate_pixels() {
  grid[[0, y as usize, x as usize]] = pixel[0] as f32;
 }
 // 2. Fetch buoy JSON data
 let buoy_resp = reqwest::blocking::get("https://api.example.com/latest_buoys?region=med")
 .unwrap().text().unwrap();
 let buoy_data: Vec<BuoyObs> = serde_json::from_str(&buoy_resp).unwrap();
 for obs in buoy_data {
 let iy = ((obs.latitude- lat_min) / 0.01).floor() as usize;
 let ix = ((obs.longitude- lon_min) / 0.01).floor() as usize;
 if iy < ny && ix < nx {
 grid[[2, iy, ix]] = obs.value;
 }
 }
 // (Fetch radar and ship data similarly; omitted for brevity...)
 // 3. Simple nearest-neighbor fill for empty grid points
 for ch in 0..grid.shape()[0] {
    let mut layer = grid.index_axis_mut(ndarray::Axis(0), ch);
 // iterate through each cell, if zero then assign nearest non-zero
 // (implementation of nearest neighbor search omitted for brevity)
 }
 // 4. Normalize channels (0-1 scaling)
 for ch in 0..grid.shape()[0] {
    let layer = grid.index_axis(ndarray::Axis(0), ch);
    let min_val = layer.iter().cloned().fold(f32::INFINITY, f32::min);
    let max_val = layer.iter().cloned().fold(f32::NEG_INFINITY, f32::max);
    if max_val > min_val {
        grid.index_axis_mut(ndarray::Axis(0), ch)
        .map_inplace(|v| *v = (*v- min_val) / (max_val- min_val));
    }
 }
 grid
 }

 This Rust function 
fetch_and_preprocess returns an 
Array3<f32> with shape [channels, ny, nx].
 In an actual application, we might also package metadata like geo-extent and resolution, but for now
 we assume consistent grid.

 2. Inference Module (Rust + ONNX Runtime or tch-rs)
 With the preprocessed data ready, we load the trained model in Rust and perform inference. We have
 two choices as noted: ONNX Runtime or tch-rs (LibTorch).
 ONNX Runtime in Rust: The onnxruntime crate provides a safe wrapper to Microsoft’s ONNX
 Runtime library . We initialize an ONNX environment, create an inference session with our
 exported model, then run the session with the input tensor.

  Steps: - Load the ONNX model (e.g.,"forecast_net.onnx" ). - Construct input tensor from ourgrid data (we likely need to add batch dimension). - Run the  session to get output.

In [ ]:
 use onnxruntime::{environment::Environment, LoggingLevel,
 GraphOptimizationLevel};
 use onnxruntime::ndarray::Array; // re-export of ndarray
 use onnxruntime::tensor::OrtOwnedTensor;
 fn run_inference(grid: Array3<f32>)-> Array4<f32> {
 // Initialize ONNX Runtime environment
 let environment = Environment::builder()
 .with_name("weather")
 .with_log_level(LoggingLevel::Warning)
 .build().expect("Failed to create ORT environment");
 // Set up session
 let mut session = environment.new_session_builder().unwrap()
 .with_optimization_level(GraphOptimizationLevel::All).unwrap()
 .with_number_threads(1).unwrap()
 .with_model_from_file("forecast_net.onnx").expect("Failed to load 
ONNX model");
 // Prepare input - our model expects shape (batch, time, channels, H, W) 
or similar.
 // If our model was exported to take a sequence, we might need to add a 
time axis.
 // For example, if we only used current state, time=1. If multiple, 
include them.
 let input_shape = grid.shape(); // e.g. (6, ny, nx)
 let channels = input_shape[0];
 let ny = input_shape[1];
 let nx = input_shape[2];
 // Convert Array3 to Array5 with batch=1 and time=1: shape (1, 1, 
channels, ny, nx)
 let input_tensor = Array::from_shape_vec(
 (1, 1, channels, ny, nx), grid.into_raw_vec()
 ).unwrap();
 // Run inference
 let outputs: Vec<OrtOwnedTensor<f32, _>> = session.run(vec!
 [input_tensor]).unwrap();
 let output_tensor = &outputs[0];
 // The output is likely shape (1, T, out_ch, ny, nx) as Array.
 let output_array = output_tensor.view().to_owned(); // copy to Array
 output_array // Array4<f32>: (T, out_ch, ny, nx) since batch=1 squeezed 
 out first dim
 }

_GeoTIFF Output and Visualization_
 The final step of the pipeline is to output the forecast maps as GeoTIFF files and optionally produce
 visualizations. We have a 4D output array (time, variable, Y, X). We can save each time slice (or variable)
 as separate GeoTIFFs or combine them (GeoTIFF can have multiple bands). For example, we might
 create one GeoTIFF per forecast hour with variables as bands, or one per variable with time as a third
 dimension (though standard GeoTIFF is 2D + bands, not temporal).

 In the above snippet, we create a single-band Float32 GeoTIFF , set the affine transform (origin and
 pixel size), define the coordinate reference system (EPSG:4326 for WGS84) , and write the pixel data.
 We invert the latitude step (positive increment downwards) by using a negative pixel height in the
 geotransform so that the GeoTIFF is oriented correctly (north-up).

 While the GeoTIFFs can be viewed in GIS software, we might also include a
 Python or Rust plotting utility for quick look. For example, using Python with Matplotlib to plot the
 numpy array with a coastline overlay (using Cartopy or Basemap), or using a Rust plotting crate or even
 generating an image with the 
image crate.

In [ ]:
 use gdal::DriverManager;
 use gdal::raster::GdalDataType;
 use gdal::spatial_ref::SpatialRef;
 fn save_geotiff(filename: &str, data: &Array2<f32>, lat_min: f64, lon_min:
 f64, res: f64) {
 let driver = DriverManager::get_driver_by_name("GTiff").unwrap();
 let (height, width) = (data.shape()[0] as usize, data.shape()[1] as
 usize);
 let dataset = driver.create_with_band_type::<f32, _>(filename, width,
 height, 1)
 .expect("Failed to create TIFF");
 // Set geotransform: tie top-left corner to (lon_min, lat_max) and set 
pixel size
 // Note: lat_max = lat_min + (height * res)
 let lat_max = lat_min + (height as f64 * res);
 let geo_transform = [lon_min, res, 0.0, lat_max, 0.0,-res];
 dataset.set_geo_transform(&geo_transform).unwrap();
 // Set projection (WGS84 lat/lon)
 let sref = SpatialRef::from_epsg(4326).unwrap();
 dataset.set_spatial_ref(&sref).unwrap();
 // Write data to band
 let band = dataset.rasterband(1).unwrap();
 band.write((0, 0), (width, height), data.as_slice().unwrap()).unwrap();
 band.flush_cache().unwrap();
 dataset.close().unwrap();
 }